# U-Net for crop-weed segmentation

This notebook contains the U-Net model for the same three-class RGB segmentation task and fixed source-image split used by the earlier notebooks.

In [ ]:
import tensorflow as tf
import numpy as np

# Use the verified CPU configuration for this TensorFlow/Metal environment.
tf.config.set_visible_devices([], 'GPU')
tf.keras.utils.set_random_seed(42)
PATCH_SIZE = 256
NUM_CLASSES = 3


## 5.1 U-Net architecture

In [ ]:
def unet_block(inputs, filters, name):
    x = tf.keras.layers.Conv2D(filters, 3, padding='same', activation='relu', name=f'{name}_conv1')(inputs)
    x = tf.keras.layers.Conv2D(filters, 3, padding='same', activation='relu', name=f'{name}_conv2')(x)
    return x

def build_unet(input_shape=(PATCH_SIZE, PATCH_SIZE, 3), num_classes=NUM_CLASSES):
    inputs = tf.keras.Input(shape=input_shape, name='rgb_input')

    # Encoder: preserve feature maps for skip connections.
    encoder_1 = unet_block(inputs, 16, 'encoder_1')
    pool_1 = tf.keras.layers.MaxPooling2D(2, name='pool_1')(encoder_1)
    encoder_2 = unet_block(pool_1, 32, 'encoder_2')
    pool_2 = tf.keras.layers.MaxPooling2D(2, name='pool_2')(encoder_2)
    encoder_3 = unet_block(pool_2, 64, 'encoder_3')
    pool_3 = tf.keras.layers.MaxPooling2D(2, name='pool_3')(encoder_3)

    bottleneck = unet_block(pool_3, 128, 'bottleneck')

    # Decoder: upsample and concatenate the matching encoder feature map.
    up_3 = tf.keras.layers.UpSampling2D(2, name='up_3')(bottleneck)
    decoder_3 = unet_block(tf.keras.layers.Concatenate(name='skip_3')([up_3, encoder_3]), 64, 'decoder_3')
    up_2 = tf.keras.layers.UpSampling2D(2, name='up_2')(decoder_3)
    decoder_2 = unet_block(tf.keras.layers.Concatenate(name='skip_2')([up_2, encoder_2]), 32, 'decoder_2')
    up_1 = tf.keras.layers.UpSampling2D(2, name='up_1')(decoder_2)
    decoder_1 = unet_block(tf.keras.layers.Concatenate(name='skip_1')([up_1, encoder_1]), 16, 'decoder_1')

    outputs = tf.keras.layers.Conv2D(num_classes, 1, padding='same', activation='softmax', name='class_probabilities')(decoder_1)
    return tf.keras.Model(inputs, outputs, name='unet')

unet = build_unet()
unet.summary()
test_output = unet(tf.zeros((1, PATCH_SIZE, PATCH_SIZE, 3)))
if tuple(test_output.shape) != (1, PATCH_SIZE, PATCH_SIZE, NUM_CLASSES):
    raise ValueError(f'Unexpected U-Net output shape: {test_output.shape}')
if not np.allclose(tf.reduce_sum(test_output, axis=-1).numpy(), 1.0, atol=1e-5):
    raise ValueError('U-Net probabilities do not sum to one.')
print('U-Net output shape:', test_output.shape)


## 5.2 U-Net training

In [ ]:
from pathlib import Path
import json
import math
from PIL import Image

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'project_plan.md').exists():
    PROJECT_ROOT = PROJECT_ROOT / 'final-project'
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
ARTIFACT_DIR = PROJECT_ROOT / 'artifacts'
FIGURE_DIR = ARTIFACT_DIR / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
split_record = json.loads((ARTIFACT_DIR / 'split.json').read_text(encoding='utf-8'))
train_images = [PROJECT_ROOT / p for p in split_record['train_images']]
val_images = [PROJECT_ROOT / p for p in split_record['validation_images']]
def mask_for(path): return path.parent.parent / 'msk' / f'{path.stem}.png'
train_masks = [mask_for(p) for p in train_images]
val_masks = [mask_for(p) for p in val_images]

def generate_patches(image_paths, mask_paths):
    for image_path, mask_path in zip(image_paths, mask_paths):
        with Image.open(image_path) as image, Image.open(mask_path) as mask:
            rgb = np.asarray(image.convert('RGB'))
            mask_rgb = np.asarray(mask.convert('RGB'))
        labels = np.full(mask_rgb.shape[:2], IGNORE_LABEL, dtype=np.uint8)
        for class_id, color in enumerate(CLASS_COLORS):
            labels[np.all(mask_rgb == color, axis=-1)] = class_id
        for y in range(0, labels.shape[0], PATCH_SIZE):
            for x in range(0, labels.shape[1], PATCH_SIZE):
                height, width = min(PATCH_SIZE, labels.shape[0] - y), min(PATCH_SIZE, labels.shape[1] - x)
                image_patch = np.zeros((PATCH_SIZE, PATCH_SIZE, 3), dtype=np.float32)
                label_patch = np.zeros((PATCH_SIZE, PATCH_SIZE), dtype=np.int32)
                weight_patch = np.zeros((PATCH_SIZE, PATCH_SIZE), dtype=np.float32)
                image_patch[:height, :width] = rgb[y:y + height, x:x + width] / 255.0
                label_patch[:height, :width] = labels[y:y + height, x:x + width]
                weight_patch[:height, :width] = 1.0
                yield image_patch, label_patch, weight_patch

def make_dataset(image_paths, mask_paths, training=False):
    dataset = tf.data.Dataset.from_generator(lambda: generate_patches(image_paths, mask_paths), output_signature=(tf.TensorSpec((PATCH_SIZE, PATCH_SIZE, 3), tf.float32), tf.TensorSpec((PATCH_SIZE, PATCH_SIZE), tf.int32), tf.TensorSpec((PATCH_SIZE, PATCH_SIZE), tf.float32)))
    if training: dataset = dataset.shuffle(64, seed=42, reshuffle_each_iteration=True)
    return dataset.batch(8)

def count_patches(paths):
    total = 0
    for path in paths:
        with Image.open(path) as image: total += math.ceil(image.height / PATCH_SIZE) * math.ceil(image.width / PATCH_SIZE)
    return total

train_steps = math.ceil(count_patches(train_images) / 8)
val_steps = math.ceil(count_patches(val_images) / 8)
train_dataset = make_dataset(train_images, train_masks, training=True).apply(tf.data.experimental.assert_cardinality(train_steps))
val_dataset = make_dataset(val_images, val_masks).apply(tf.data.experimental.assert_cardinality(val_steps))
print('Training steps:', train_steps, 'Validation steps:', val_steps)


In [ ]:
unet.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss=tf.keras.losses.SparseCategoricalCrossentropy())
UNET_CHECKPOINT = ARTIFACT_DIR / 'unet_best.keras'
unet_callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, mode='min', restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ModelCheckpoint(UNET_CHECKPOINT, monitor='val_loss', mode='min', save_best_only=True, verbose=1),
]
UNET_MAX_EPOCHS = 30
unet_history = unet.fit(train_dataset, validation_data=val_dataset, epochs=UNET_MAX_EPOCHS, callbacks=unet_callbacks, shuffle=False)
history_values = {key: [float(value) for value in values] for key, values in unet_history.history.items()}
(ARTIFACT_DIR / 'unet_history.json').write_text(json.dumps(history_values, indent=2) + '\n', encoding='utf-8')
import matplotlib.pyplot as plt
figure, axis = plt.subplots(figsize=(7, 4))
axis.plot(history_values['loss'], label='Training loss')
axis.plot(history_values['val_loss'], label='Validation loss')
axis.set(xlabel='Epoch', ylabel='Sparse categorical cross-entropy', title='U-Net training curves')
axis.legend(); axis.grid(alpha=0.3); figure.tight_layout()
figure.savefig(FIGURE_DIR / 'unet_curves.png', dpi=150)
plt.show(); plt.close(figure)
print('Saved:', UNET_CHECKPOINT)
